# 114/611 In-class exercise: Prompt Engineering for QA

In this exercise, you will construct specific prompts for a set of expected answer types, and learn about different evaluation methods.

We will use `gpt-4.1-mini`, so be sure you have an OpenAI key, which you may create on the [CMU AI Gateway](https://ai-gateway.andrew.cmu.edu/ui/?login=success&page=api-keys).

*(**Warning**: Each student has $50 API credit throughout the semester, so please keep track of your usage!)*

Please submit your notebook in a completed state, i.e. run the notebook to completion, and don't erase the content of the output cells, especially in the "📋**Results**" sections. Remember to answer the "✍️**Reflection**" sections in the notebook!



## Challenge of QA: How do we evaluate the answers?

When humans ask and answer questions, we rarely stick to strict formats or predefined options like in multiple-choice tests. Instead, we interpret the information we receive and decide whether it makes sense or satisfies our intent.

Dealing QA systems is more difficult. Even for factual questions, it can be difficult to assess the answers when no clear constraints or examples are provided, where methods like exact match may fail. This is more complicated for questions that require free-form generation.

In this notebook, we focus on factual QA, and in this section, we'll explore some evaluation methods based on the example below.

In [1]:
# DO NOT CHANGE THIS CELL
question = "What are the benefits of exercise?"
expectedAnswer = [
    "improves heart health",
    "strengthens muscles",
    "boosts mental health"
]
possible_answers = {
    "Answer 1": "Exercise improves heart health, strengthens muscles, and boosts mental health.",
    "Answer 2": "Exercise improves heart health but often harms mental health due to stress.",
    "Answer 3": "Exercise keeps the body strong and improves mood.",
    "Answer 4": "Exercise is tiring, weakens the body, and causes stress.",
}

### Method 1 : Soft Match (Factoid and Factoid List Answers)



**Soft-match** is a string-matching-based evaluation method that relaxes the constraint of exact match. It can also handle cases where multiple answers are acceptable.

For a single expected answer (**factoid**), we check whether the expected answer appears in the model's output.
- If it does not occur, the score is 0.
- If it does, the score is computed as the ratio of the number of characters in the expected answer to the number of characters in the generated answer, which penalizes redundant words.

For multiple expected answers (**factoid list**), we assume the generated output is a comma-separated list of distinct factoids. In this case, we sum the scores for each expected factoid, and apply a discount factor to account for the extra punctuation or separators in the list.

In [2]:
# DO NOT CHANGE THIS CELL
import string

def softMatch ( generatedString , expectedString, discount ):
  if expectedString in generatedString:
    score = len(expectedString) / (len(generatedString) - discount)
  else:
    score = 0
  return score

def evaluate_soft_match(question, answer, expectedAnswer):
    if isinstance(expectedAnswer, list):
        total = 0
        discount = 2*len(expectedAnswer)
        for exp in expectedAnswer:
            total += softMatch(answer, exp, discount)
            total = min(total, 1.0)
        print("\nSoftmatch Score:", f"{total:.2f}")
    else:
        score = softMatch(answer, expectedAnswer, 0)
        print("\nSoftmatch Score:", f"{score:.2f}")


In [3]:
# DO NOT CHANGE THIS CELL
# Soft Match Exploration Cell
print(f"Question: {question}\n")
for label, ans in possible_answers.items():
    print(f"--- {label} ---")
    evaluate_soft_match(question, ans, expectedAnswer)
    print(f"Answer: {ans}\n")


Question: What are the benefits of exercise?

--- Answer 1 ---

Softmatch Score: 0.83
Answer: Exercise improves heart health, strengthens muscles, and boosts mental health.

--- Answer 2 ---

Softmatch Score: 0.30
Answer: Exercise improves heart health but often harms mental health due to stress.

--- Answer 3 ---

Softmatch Score: 0.00
Answer: Exercise keeps the body strong and improves mood.

--- Answer 4 ---

Softmatch Score: 0.00
Answer: Exercise is tiring, weakens the body, and causes stress.



#### ✍️ **Reflection**

Please use the text cell below to answer the following question:

What are the strengths and weaknesses of SoftMatch? Give at least one strength and one weakness; refer to the examples above for in order to illustrate your points.

**Your Answer**

A strength of SoftMatch is that it is simple, fast, and cheap. It gives a high score when the expected facts appear clearly in the answer, such as Answer 1, which includes all three expected benefits almost exactly. A weakness is that it relies too much on exact wording. For example, Answer 3 is partly correct in meaning because "keeps the body strong" and "improves mood" are close to the gold answers, but SoftMatch still scores it low because the exact phrases are missing. It also cannot reason about contradictions, so Answer 2 still gets some credit for mentioning heart health even though it incorrectly says exercise harms mental health.


### Method 2: LLM as a Judge

Another way to evaluate answers is by using a **LLM as a judge**, which is a popular method recently. In real-world applications, we often choose the best affordable LLM available to serve as the evaluator.

To make LLMs judge, we have to provide explicit evaluation instructions to the model. In this notebook, we ask the LLM to rate each answer on a Likert scale from 0 to 5, and then normalize the score to a 0-1 range so it can be compared directly with the soft-match score.

We also track the evaluation cost based on the data stored in the `PRICE` dictionary.

In [4]:
# Define prompts and price of the model we choose

JUDGE_PROMPT = """# Instruction
You will be given a question, gold answer, and system answer.
Your task is to provide a 'total rating' scoring
how well the system answer matches the gold answer for the question.
Give your answer as an integer on a scale of 0 to 5, where
0 means that the system answer does not match the gold answer at all,
and 5 means that the system answer matches the gold answer.

Provide your feedback as follows:

# Feedback
Rationale: (your thinking process)
Total rating: (your rating, as an integer from 0 to 5)"""

TASK_PROMPT = """# Task
Now here are the question and answer.
Question: {question}
Gold Answer: {gold_answer}
System Answer: {system_answer}

# Feedback
Rationale: """

PRICE = {
     'input_tokens': 0.40/1e6,   
    'output_tokens': 1.60/1e6   
}

In [5]:
# DO NOT CHANGE THIS CELL
from collections import defaultdict

def parse(text):
    """
    Parse an output, assuming the following output format:
    xxx Total rating: y

    """
    output = 0
    if 'Total rating:' in text:
        splits = text.split('Total rating:')
        score = splits[-1].strip()
        if score.isdigit():
            output = int(score)
        else:
            print(f"Error: score cannot be converted to integer.")
    else:
        print(f"Error: output does not follow the specified format.")

    return output

def evaluate_llm_as_a_judge(judge, examples: list[dict[str, str]]):
    """
    Given a judge and examples, print out
    * the average score
    * the api cost for the evaluation
    and return the score scaled to 0-1

    """
    scores = []
    usage = defaultdict(int)
    for example in examples:

        response = judge.responses.create(
            model="gpt-4.1-mini",
            instructions=JUDGE_PROMPT,
            input=TASK_PROMPT.format(
                question=example['question'],
                gold_answer=example['gold_answer'],
                system_answer=example['system_answer']
                )
        )
        output_raw = response.output[0].content[0].text
        scores.append(parse(output_raw))
        usage['input_tokens'] += response.usage.input_tokens
        usage['output_tokens'] += response.usage.output_tokens

    cost = sum(usage[k]*v for k, v in PRICE.items())

    avg = sum(scores)/len(scores) / 5 # normalize
    print(f"\nLLM Likert Score (normalized): {avg:.2f} (Cost: {cost:.4f} USD)")
    return


In [6]:
# DO NOT CHANGE THIS CELL
# Get the user's OpenAI key and create a client model to be used for answering
# questions as well as judging answers.
import openai
import getpass
llm = openai.OpenAI(
    api_key=os.getenv("API_KEY", ""),
    base_url="https://ai-gateway.andrew.cmu.edu/"
)


llm = openai.OpenAI(
    api_key="******",
    base_url="https://ai-gateway.andrew.cmu.edu/"
)

In [1]:
import openai
client = openai.OpenAI(
      api_key=os.getenv("API_KEY", ""),
      base_url='https://joseph-coding-agent-test.services.ai.azure.com/api/projects/proj-default'
  )
for m in client.models.list():
    print(m.id)

claude-3-haiku-20240307
claude-3-5-sonnet-20241022
claude-3-7-sonnet-20250219-v1:0
llama3-2-11b-instruct
llama3-2-90b-instruct
meta.llama3-1-8b-instruct-v1:0
claude-sonnet-4-20250514-v1:0
claude-opus-4-20250514-v1:0
claude-haiku-4-5-20251001-v1:0
us.anthropic.claude-opus-4-6-v1
us.anthropic.claude-haiku-4-5-20251001-v1:0
us.anthropic.claude-sonnet-4-6
gemini-2.5-pro
gemini-2.5-flash
gemini/gemini-3.1-flash-lite-preview
gemini/gemini-3.1-pro-preview
gpt-5
gpt-5.4-mini
gpt-5.4-nano
gpt-5.4-pro
gpt-4.1-mini
o1-mini-2024-09-12
azure/text-embedding-3-small
gpt-5-mini
gpt-4o-transcribe
gpt-5.4


In [7]:
# DO NOT CHANGE THIS CELL
# LLM as a judge Exploration Cell
print(f"Question: {question}\n")
gold_answer = ", ".join(expectedAnswer)
for label, ans in possible_answers.items():
  print(f"--- {label} ---")
  evaluate_llm_as_a_judge(llm, [{"question": question, "system_answer": ans, "gold_answer": gold_answer}])
  print(f"Answer: {ans}\n")

Question: What are the benefits of exercise?

--- Answer 1 ---

LLM Likert Score (normalized): 1.00 (Cost: 0.0002 USD)
Answer: Exercise improves heart health, strengthens muscles, and boosts mental health.

--- Answer 2 ---

LLM Likert Score (normalized): 0.40 (Cost: 0.0003 USD)
Answer: Exercise improves heart health but often harms mental health due to stress.

--- Answer 3 ---

LLM Likert Score (normalized): 0.60 (Cost: 0.0002 USD)
Answer: Exercise keeps the body strong and improves mood.

--- Answer 4 ---

LLM Likert Score (normalized): 0.00 (Cost: 0.0002 USD)
Answer: Exercise is tiring, weakens the body, and causes stress.



#### ✍️ **Reflection**

Please use the text cell below to answer the following question:

Comparing to SoftMatch, how is the performance of LLM as a judge? Give at least one strength and one weakness; refer to the examples above in order to illustrate your points.

**Your Answer**

Compared with SoftMatch, LLM as a judge is better at recognizing meaning instead of exact wording. For example, an answer like "Exercise keeps the body strong and improves mood" can still receive partial credit because the judge can connect it to "strengthens muscles" and "boosts mental health," while SoftMatch gives a much lower score. Another strength is that the judge can penalize contradictions, so Answer 2 should score lower because it includes the incorrect claim that exercise harms mental health. The weakness is that LLM judging is slower, costs money, and can vary across runs. It also depends heavily on prompt quality, so unclear instructions can lead to inconsistent ratings.


## Hands-on: How do we get better answers from LLMs?

### QA Data

These are the question and answer pairs, while `type` indicates the format of the expected answer.
Therefore, some questions are included more than once, with different `type` and `answer`.

In [8]:
# DO NOT CHANGE THIS CELL.

q1 = {
    "type": "LAST_NAME",
    "question": "Who are the first three presidents of United States?",
    "answer": ["Washington", "Adams", "Jefferson"],
}
q2 = {
    "type": "FULL_NAME",
    "question": "Who are the first three presidents of United States?",
    "answer": ["George Washington", "John Adams", "Thomas Jefferson"],
}
q3 = {
    "type": "FIRST_NAME",
    "question": "Who are the first three presidents of United States?",
    "answer": ["George", "John", "Thomas"],
}
q4 = {
    "type": "LAST_NAME",
    "question": "Who is the current president of Carnegie Mellon?",
    "answer": "Jahanian"
}
q5 = {
    "type": "FULL_NAME",
    "question": "Who is the current president of Carnegie Mellon?",
    "answer": "Farnam Jahanian"
}
q6 = {
    "type": "FIRST_NAME",
    "question": "Who is the current president of Carnegie Mellon?",
    "answer": "Farnam"
}
q7 = {
    "type": "MONTH",
    "question": "When was Charles Dickens born?",
    "answer": "February"
}
q8 = {
    "type": "DATE",
    "question": "When was Charles Dickens born?",
    "answer": "February 7, 1812"
}
q9 = {
    "type": "LIST_OF_STEPS",
    "question": "How do I get a PA driver's license?",
    "answer": ["Get a medical exam","Study the manual","Gather required documents","Take the knowledge and vision tests","Receive your permit and practice","Schedule and pass the road test","Get your license at a Photo License Center"]
}

q10 = {
    "type": "LIST_OF_STEPS",
    "question": "How do I get a US passport?",
    "answer": ["Complete and print Form DS-11", "Gather proof of U.S. citizenship and identity" , "Get a passport photo", "Make an appointment at an acceptance facility to submit your application in person"]
}

### Answering question with LLMs

Now that we have the data and the LLM API ready, let's start answering questions using the basic setup!

In the `answerQuestion` function, you can choose which **question**, **prompt format**, and **evaluation method** to use in order to test how the LLM performs.
In the basic setup, where `type_specific_prompts` is empty, all question types use the general prompt description.
The full prompt and response are then generated using `llm.chat.completions.create`, and then answer is evaluated automatically.

**Note**: Subsequent calls to the model with the same prompt and question may produce slightly different outputs. Don’t worry — you can report a single representative result in the output cells below.

In [9]:
# DO NOT CHANGE THIS CELL
# Define the dict used to store prompts per answer type.
type_specific_prompts = {}

In [ ]:
# DO NOT CHANGE THIS CELL

def answerQuestion ( input , prompt_type="general", evaluation_type="softmatch"):
  answerType = input["type"]
  question = input["question"]
  expectedAnswer = input["answer"]
  if prompt_type == "general":
    prompt = "Answer the following question."
  else: # use type specific prompts
    prompt = type_specific_prompts.get( answerType , "Answer the following question.")

  print("Answer Type: "+ answerType )
  print("\nPrompt: " + prompt )
  print("\nQuestion: " + question )

  response = llm.chat.completions.create(
    model="gpt-4.1-mini",
    messages = [
        { "role": "system", "content": f"{prompt}" },
        { "role": "user", "content":  f"Question:\n{question}"}

    ]
  )
  answer = response.choices[0].message.content
  print("\nAnswer:\n" + answer )
  print("\nExpected Answer:\n" + str(expectedAnswer) )

  if evaluation_type == "softmatch":
    evaluate_soft_match( question, answer, expectedAnswer )
  else: # eval with llm as a judge
    evaluate_llm_as_a_judge(llm, [{"question": question, "system_answer": answer, "gold_answer": expectedAnswer}])


In [11]:
# Try answering a single question like this!
answerQuestion( q1 )

Answer Type: LAST_NAME

Prompt: Answer the following question.

Question: Who are the first three presidents of United States?

Answer:
The first three presidents of the United States are:

1. George Washington (1789–1797)  
2. John Adams (1797–1801)  
3. Thomas Jefferson (1801–1809)

Expected Answer:
['Washington', 'Adams', 'Jefferson']

Softmatch Score: 0.17


#### Results (to compare with the next section)

In [12]:
# Change this cell if needed. The output should include the results of all questions.

# NOTE: use indices (0–9), not q1/q2 objects
use_llm_as_a_judge = [8, 9] # Use LLM as a judge for open-ended step lists.
for i, question in enumerate([q1, q2, q3, q4, q5, q6, q7, q8, q9, q10]):
  print(f"---------------- Question {i+1} ----------------")
  if i in use_llm_as_a_judge:
    answerQuestion( question , evaluation_type="llm_as_a_judge" )
  else:
    answerQuestion( question )

---------------- Question 1 ----------------
Answer Type: LAST_NAME

Prompt: Answer the following question.

Question: Who are the first three presidents of United States?

Answer:
The first three presidents of the United States are:

1. George Washington (1789–1797)  
2. John Adams (1797–1801)  
3. Thomas Jefferson (1801–1809)

Expected Answer:
['Washington', 'Adams', 'Jefferson']

Softmatch Score: 0.17
---------------- Question 2 ----------------
Answer Type: FULL_NAME

Prompt: Answer the following question.

Question: Who are the first three presidents of United States?

Answer:
The first three presidents of the United States are:

1. George Washington (1789–1797)  
2. John Adams (1797–1801)  
3. Thomas Jefferson (1801–1809)

Expected Answer:
['George Washington', 'John Adams', 'Thomas Jefferson']

Softmatch Score: 0.30
---------------- Question 3 ----------------
Answer Type: FIRST_NAME

Prompt: Answer the following question.

Question: Who are the first three presidents of United 

### Prompt Engineering 1: Answer Format

After experimenting with `answerQuestion`, you may have noticed some limitations of the general prompt. For the next step, you'll tune the prompts to **produce answers in the correct format** for each question type!

Your goal is to improve performance by **adjusting only the prompts below**. Try to design prompts so that the generated output is as close as possible to the expected answer, given what the LLM is capable of producing.

*(You don't need to add RAG contexts or external information; the focus here is on using prompts to control the level of detail and structure in the output.)*

**Hint**: Work through the questions one by one. After updating a prompt, re-run the corresponding evaluation cell to see how your new prompt affects the model's response.

In [13]:
# TODO: Change the prompts!

# Names
type_specific_prompts["LAST_NAME"] = "Answer the question using ONLY the person's last name. If there are multiple people, return only the last names in chronological order, separated by commas, with no numbering and no extra words."
type_specific_prompts["FULL_NAME"] = "Answer the question using ONLY full names. If there are multiple people, return only the full names in chronological order, separated by commas, with no numbering and no extra words."
type_specific_prompts["FIRST_NAME"] = "Answer the question using ONLY the person's first name. If there are multiple people, return only the first names in chronological order, separated by commas, with no numbering and no extra words."

# Dates
type_specific_prompts["MONTH"] = "Answer using ONLY the month name, with no day, year, or extra words."
type_specific_prompts["DATE"] = "Answer using ONLY the full date in the format Month Day, Year, with no extra words."

# Steps
type_specific_prompts["LIST_OF_STEPS"] = "Answer with a short comma-separated list of the main steps only. Do not use numbering, bullet points, explanations, or extra details."

In [14]:
# Try answering questions one by one! Index to format type:
# LAST_NAME: q1, q4
# FULL_NAME: q2, q5
# FIRST_NAME: q3, q6
# MONTH / DATE: q7, q8
# LIST_OF_STEPS: q9, q10
answerQuestion( q1, prompt_type="specific")

Answer Type: LAST_NAME

Prompt: Answer the question using ONLY the person's last name. If there are multiple people, return only the last names in chronological order, separated by commas, with no numbering and no extra words.

Question: Who are the first three presidents of United States?

Answer:
Washington, Adams, Jefferson

Expected Answer:
['Washington', 'Adams', 'Jefferson']

Softmatch Score: 1.00


#### 📋**Results**

In [15]:
# Change this cell if needed. The output should include the results of all questions.

# NOTE: use indices (0–9), not q1/q2 objects
use_llm_as_a_judge = [8, 9] # Use LLM as a judge for open-ended step lists.
for i, question in enumerate([q1, q2, q3, q4, q5, q6, q7, q8, q9, q10]):
  print(f"---------------- Question {i+1} ----------------")
  if i in use_llm_as_a_judge:
    answerQuestion( question , prompt_type="specific" , evaluation_type="llm_as_a_judge" )
  else:
    answerQuestion( question , prompt_type="specific")

---------------- Question 1 ----------------
Answer Type: LAST_NAME

Prompt: Answer the question using ONLY the person's last name. If there are multiple people, return only the last names in chronological order, separated by commas, with no numbering and no extra words.

Question: Who are the first three presidents of United States?

Answer:
Washington, Adams, Jefferson

Expected Answer:
['Washington', 'Adams', 'Jefferson']

Softmatch Score: 1.00
---------------- Question 2 ----------------
Answer Type: FULL_NAME

Prompt: Answer the question using ONLY full names. If there are multiple people, return only the full names in chronological order, separated by commas, with no numbering and no extra words.

Question: Who are the first three presidents of United States?

Answer:
George Washington, John Adams, Thomas Jefferson

Expected Answer:
['George Washington', 'John Adams', 'Thomas Jefferson']

Softmatch Score: 1.00
---------------- Question 3 ----------------
Answer Type: FIRST_NAME



#### ✍️ **Reflection**

Please use the text cell below to answer the following question:

What changes did you observe after applying the type-specific formatting prompts?
Were there any challenges or difficulties in designing these prompts?

**Your Answer**

After applying type-specific formatting prompts, the answers should become much closer to the expected format for names and dates. Instead of full sentences, the prompts encourage outputs like only last names, only first names, only the month, or only the exact date. That should improve performance because the model is no longer adding extra explanation that SoftMatch penalizes.

One challenge was that list-of-steps questions are harder to control exactly. Even with a better prompt, the model may still add extra details or choose slightly different wording, which can hurt SoftMatch even when the answer is reasonable. Another difficulty is that prompt design can improve structure, but it does not guarantee the model will match the gold wording exactly.

### Prompt Engineering 2: LLM as a Judge

Formatting helps us get better answers from LLMs, but evaluating those answers is equally important. The clearer the instructions, the more stable and reliable the LLM's scores will be.

Looking at the prompt and the evaluation scores above, can you **revise the judge prompt and/or task prompt** to improve the LLM-as-a-judge's performance?

In [16]:
# TODO: Change the prompts! This is copied from above

JUDGE_PROMPT = """# Instruction
You will be given a question, a gold answer, and a system answer.
Rate how well the system answer matches the gold answer for the question.

Scoring rubric:
- 5: Fully correct. Same facts and requested format, or a harmless formatting difference only.
- 4: Mostly correct. Minor missing detail or slightly different formatting, but meaning is correct.
- 3: Partially correct. Some correct content, but important details are missing, extra, or not in the requested form.
- 2: Slightly correct. Small overlap with the gold answer, but mostly incomplete or partly wrong.
- 1: Almost entirely incorrect. Very little meaningful overlap.
- 0: Completely incorrect, irrelevant, or contradicts the gold answer.

Evaluation rules:
- Focus on factual correctness first, then format.
- For name questions, accept equivalent subsets only if they match the requested type such as first name, last name, or full name.
- For list questions, compare whether the main items or steps match even if wording differs slightly.
- Do not reward extra explanation if the gold answer expects a short formatted answer.
- Return exactly the format below.

# Feedback
Rationale: <one short sentence>
Total rating: <integer 0-5>"""

TASK_PROMPT = """# Task
Question: {question}
Gold Answer: {gold_answer}
System Answer: {system_answer}

Judge the system answer using the rubric above.

# Feedback
Rationale: """


#### 📋**Results**

In [17]:
# Change this cell if needed. The output should include the results of questions that use LLM as a judge.

# NOTE: use indices (0–9) for questions
use_llm_as_a_judge = [8, 9]
for i, question in enumerate([q1, q2, q3, q4, q5, q6, q7, q8, q9, q10]):
  if i not in use_llm_as_a_judge: continue
  print(f"---------------- Question {i+1} ----------------")
  answerQuestion( question , prompt_type="specific" , evaluation_type="llm_as_a_judge" )

---------------- Question 9 ----------------
Answer Type: LIST_OF_STEPS

Prompt: Answer with a short comma-separated list of the main steps only. Do not use numbering, bullet points, explanations, or extra details.

Question: How do I get a PA driver's license?

Answer:
pass the learner's permit test, complete required learner's permit holding period, complete driver's education and behind-the-wheel training, schedule and pass the road test, submit required documents and fees, obtain driver's license

Expected Answer:
['Get a medical exam', 'Study the manual', 'Gather required documents', 'Take the knowledge and vision tests', 'Receive your permit and practice', 'Schedule and pass the road test', 'Get your license at a Photo License Center']

LLM Likert Score (normalized): 0.80 (Cost: 0.0003 USD)
---------------- Question 10 ----------------
Answer Type: LIST_OF_STEPS

Prompt: Answer with a short comma-separated list of the main steps only. Do not use numbering, bullet points, explanat

#### ✍️ **Reflection**

Please use the text cell below to answer the following question:

What observations guided your prompt design for the LLM-as-a-judge method?
Did the prompts work as you expected?
Were there any challenges in designing these prompts?

**Your Answer**

My prompt design was guided by the observation that the original judge prompt was too vague about partial credit, paraphrases, and formatting differences. I revised it to include a clearer rubric and explicit rules for names, lists, and extra explanation, because those were the cases where surface matching and human judgment differed most.

The prompts should work better because they tell the judge to focus on factual correctness first and then consider formatting. A challenge is that LLM judging can still be somewhat inconsistent across runs, and prompt wording matters a lot. Even with a better rubric, the judge may still occasionally over-reward plausible but incomplete answers or be stricter than expected.


### Prompt Engineering 3: LLM as a Classifier (Few-shot)

What’s going on?

LLMs often rely on keywords instead of true intent.  
This can lead to incorrect classifications when a request is slightly ambiguous.

In this task, you will:
1. First try classification **without examples**
2. Then add **few-shot examples**
3. Observe how the prediction(classification) improves

In [18]:
# DO NOT CHANGE THIS CELL
q11 = {
    "type": "TICKET_CLASSIFIER",
    "question": "Request: I logged in successfully, but now I can't access my account settings and it keeps saying 'invalid session' even after retrying.",
    "answer": "Account"
}

In [19]:
# DO NOT CHANGE THIS CELL
# Baseline (no examples)
# Use a simple prompt without examples 
type_specific_prompts["TICKET_CLASSIFIER"] = """
Classify the request into one of: Billing, Technical, Account.
"""
answerQuestion(q11, prompt_type="specific")

Answer Type: TICKET_CLASSIFIER

Prompt: 
Classify the request into one of: Billing, Technical, Account.


Question: Request: I logged in successfully, but now I can't access my account settings and it keeps saying 'invalid session' even after retrying.

Answer:
Technical

Expected Answer:
Account

Softmatch Score: 0.00


Now, we will improve the model by adding **examples**.

You are given a few example requests and their correct labels.  

**Your task is to use these examples below as a guide.**  


Training Examples

- Request: I tried logging into my account after resetting my password, but it keeps saying credentials are incorrect even though I’m sure they are right.  
  Label: Account  

- Request: I noticed that my credit card was charged twice this month for the same subscription, and I don’t see any explanation in my billing history.  
  Label: Billing  

- Request: Whenever I open the app, it loads for a few seconds and then crashes back to the home screen without showing any error message.  
  Label: Technical  


Now complete the prompt in the next cell.

In [20]:
# TODO: Change the prompt! Add examples to the prompt to see if it changes performance.
type_specific_prompts["TICKET_CLASSIFIER"] = """
Classify the request into one of: Billing, Technical, Account.
Return ONLY the label.

Examples:

Request: I tried logging into my account after resetting my password, but it keeps saying credentials are incorrect even though I am sure they are right.
Label: Account

Request: I noticed that my credit card was charged twice this month for the same subscription, and I do not see any explanation in my billing history.
Label: Billing

Request: Whenever I open the app, it loads for a few seconds and then crashes back to the home screen without showing any error message.
Label: Technical
"""
answerQuestion(q11, prompt_type="specific")

Answer Type: TICKET_CLASSIFIER

Prompt: 
Classify the request into one of: Billing, Technical, Account.
Return ONLY the label.

Examples:

Request: I tried logging into my account after resetting my password, but it keeps saying credentials are incorrect even though I am sure they are right.
Label: Account

Request: I noticed that my credit card was charged twice this month for the same subscription, and I do not see any explanation in my billing history.
Label: Billing

Request: Whenever I open the app, it loads for a few seconds and then crashes back to the home screen without showing any error message.
Label: Technical


Question: Request: I logged in successfully, but now I can't access my account settings and it keeps saying 'invalid session' even after retrying.

Answer:
Account

Expected Answer:
Account

Softmatch Score: 1.00


#### ✍️ **Reflection**

Please use the text cell below to answer the following question:

What label did the model predict before adding examples?  
What label did it predict after adding examples?  
Why did the examples help improve the result?

**Your Answer**

Before adding examples, the model predicted `Technical`. After adding the few-shot examples, it predicted `Account`.

The examples help because they show the model how to interpret the intent of the request instead of reacting only to keywords like `invalid session`. The few-shot prompt makes it clearer that login, credentials, and account-access problems belong to `Account`, while crashes and app failures belong to `Technical`.


In [1]:
# Google Gemini — OpenAI-compatible endpoint
gemini_client = openai.OpenAI(
    api_key=os.getenv("API_KEY", ""),
    base_url='https://generativelanguage.googleapis.com/v1beta/openai/'
)

print("=== Google Gemini Models ===")
gemini_models = list(gemini_client.models.list())
print(f"Total: {len(gemini_models)}\n")
for m in sorted(gemini_models, key=lambda x: x.id):
    print(m.id)

NameError: name 'openai' is not defined

In [2]:
# Together AI — OpenAI-compatible endpoint
import openai

together_client = openai.OpenAI(
    api_key=os.getenv("API_KEY", ""),
    base_url='https://api.together.xyz/v1'
)

print("=== Together AI Models ===")
models = list(together_client.models.list())
print(f"Total: {len(models)}\n")
for m in sorted(models, key=lambda x: x.id):
    print(m.id)

=== Together AI Models ===


AttributeError: 'list' object has no attribute '_set_private_attributes'

## Extra: Available Models — Together AI & Google Gemini

Testing which models we have access to via the Together AI and Google Gemini APIs.

In [3]:
from together import Together

client = Together(api_key=os.getenv("API_KEY", ""))

models = client.models.list()
for model in models:
    print(model.id)

zai-org/GLM-5.1
MiniMaxAI/MiniMax-M2.7
google/gemma-4-31B-it
Qwen/Qwen3.5-397B-A17B
MiniMaxAI/MiniMax-M2.5
moonshotai/Kimi-K2.5
openai/gpt-oss-120b
openai/gpt-oss-20b
zai-org/GLM-5
deepseek-ai/DeepSeek-R1
deepseek-ai/DeepSeek-V3.1
Qwen/Qwen3.5-9B
Qwen/Qwen3-Coder-Next-FP8
Qwen/Qwen3-Coder-480B-A35B-Instruct-FP8
Qwen/Qwen3-235B-A22B-Instruct-2507-tput
Qwen/Qwen2.5-7B-Instruct-Turbo
meta-llama/Llama-3.3-70B-Instruct-Turbo
meta-llama/Meta-Llama-3-8B-Instruct-Lite
mistralai/Mistral-Small-24B-Instruct-2501
google/gemma-3n-E4B-it
hexgrad/Kokoro-82M
canopylabs/orpheus-3b-0.1-ft
openai/whisper-large-v3
black-forest-labs/FLUX.1-schnell
black-forest-labs/FLUX.1-krea-dev
black-forest-labs/FLUX.1-kontext-pro
black-forest-labs/FLUX.1-kontext-max
black-forest-labs/FLUX.2-dev
black-forest-labs/FLUX.2-flex
black-forest-labs/FLUX.2-pro
black-forest-labs/FLUX.2-max
black-forest-labs/FLUX.1.1-pro
meta-llama/Llama-Guard-4-12B
intfloat/multilingual-e5-large-instruct
arize-ai/qwen-2-1.5b-instruct
LiquidAI/L

In [4]:
from google import genai

client = genai.Client(api_key=os.getenv("API_KEY", ""))

models = client.models.list()
for model in models:
    print(model.name)

/Users/mac/Documents/cmu/spring26/intro-deep-learning/project/stategen/.venv/lib/python3.9/site-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
/Users/mac/Documents/cmu/spring26/intro-deep-learning/project/stategen/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/mac/Documents/cmu/spring26/intro-deep-learning/project/stategen/.venv/lib/python3.9/site-packages/google/oauth2/__init__.py:40: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update googl

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gem

In [1]:
import os
from openai import AzureOpenAI

endpoint = "https://stategen.openai.azure.com/"
model_name = "gpt-4o"
deployment = "gpt-4o-stategen"

subscription_key=os.getenv("AZURE_OPENAI_API_KEY", "")
api_version = "2024-12-01-preview"

client = AzureOpenAI(
    api_version=api_version,
    azure_endpoint=endpoint,
    api_key=subscription_key,
)

response = client.chat.completions.create(
    messages=[
        {
            "role": "system",
            "content": "You are a helpful assistant.",
        },
        {
            "role": "user",
            "content": "I am going to Paris, what should I see?",
        }
    ],
    max_tokens=4096,
    temperature=1.0,
    top_p=1.0,
    model=deployment
)

print(response.choices[0].message.content)

Paris is a city brimming with history, art, and culture, so there’s no shortage of things to see and do. Here's a list of must-visit spots and activities to make the most of your visit:

---

### **Iconic Landmarks**
1. **Eiffel Tower** - The symbol of Paris! Visit during the day or evening when it sparkles on the hour. You can climb or take an elevator to the top for stunning views.
2. **Louvre Museum** - See world-famous art pieces like the *Mona Lisa* and *The Venus de Milo*. The museum itself is an architectural marvel.
3. **Notre Dame Cathedral** - While currently under restoration (post-2019 fire), the facade and surrounding area are stunning, and Île de la Cité is worth exploring.
4. **Arc de Triomphe** - Climb to the top for breathtaking views of the Champs-Élysées and the 12 radiating avenues.
5. **Sainte-Chapelle** - A Gothic masterpiece known for its stunning stained-glass windows.
6. **Sacré-Cœur Basilica** - Located in Montmartre, it offers incredible views over Paris and 